# 08 — CatBoost, Advanced Boosting (Stage 10)
CatBoost uses ordered boosting and symmetric (oblivious) trees, which reduce prediction shift and overfitting on small data. Tuned: iterations, depth, learning_rate, l2_leaf_reg, loss_function (MultiClass vs MultiClassOneVsAll), auto_class_weights.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
FIG = ROOT / 'figures'; REP = ROOT / 'reports'

In [2]:
summary = pd.read_csv(REP / 'tuning' / 'tuning_summary.csv').set_index('model')
row = summary.loc['catboost']
print('Search             :', row['search'], f"({row['n_candidates']} candidates)")
print('CV Macro F1        :', f"{row['cv_macro_f1_mean']:.4f} ± {row['cv_macro_f1_std']:.4f}")
print('Train Macro F1     :', f"{row['train_macro_f1_mean']:.4f}")
print('Train - CV gap     :', f"{row['train_minus_cv_gap']:+.4f}")
print('Best parameters    :', row['best_params'])

Search             : RandomizedSearchCV (20 candidates)
CV Macro F1        : 0.7766 ± 0.0304
Train Macro F1     : 1.0000
Train - CV gap     : +0.2234
Best parameters    : {'model__auto_class_weights': 'SqrtBalanced', 'model__depth': 8, 'model__iterations': 466, 'model__l2_leaf_reg': 1.0310149462998892, 'model__learning_rate': 0.25653418421266855, 'model__loss_function': 'MultiClassOneVsAll'}


Fold-wise Macro F1 (same 5 stratified folds for every model):

In [3]:
folds = pd.read_csv(REP / 'tuning' / 'cv_fold_scores.csv', index_col=0)
folds.loc[[i for i in folds.index if i.startswith('CatBoost')]]

,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
CatBoost (initial),0.7636,0.7534,0.7413,0.7698,0.7476,0.7551,0.0104
CatBoost,0.8062,0.7423,0.7813,0.8122,0.7411,0.7766,0.0304


Top 10 candidates of the search, ranked by mean CV Macro F1:

In [4]:
res = pd.read_csv(REP / 'tuning' / 'catboost_search_results.csv')
cols = [c for c in res.columns if c.startswith('param_')] + ['mean_test_score', 'std_test_score', 'mean_train_score', 'mean_fit_time']
res.sort_values('rank_test_score')[cols].head(10).round(4)

,param_model__auto_class_weights,param_model__depth,param_model__iterations,param_model__l2_leaf_reg,param_model__learning_rate,param_model__loss_function,mean_test_score,std_test_score,mean_train_score,mean_fit_time
6,SqrtBalanced,8,466,1.0310,0.2565,MultiClassOneVsAll,0.7766,0.0304,1.0000,8.6368
14,SqrtBalanced,6,798,5.0916,0.1440,MultiClass,0.7756,0.0312,1.0000,5.0204
5,SqrtBalanced,6,866,9.6212,0.0708,MultiClass,0.7728,0.0304,0.9855,5.6946
0,SqrtBalanced,7,570,5.3950,0.1012,MultiClass,0.7721,0.0326,0.9973,6.2240
8,SqrtBalanced,7,808,6.8107,0.0320,MultiClass,0.7693,0.0335,0.9746,8.8242
15,SqrtBalanced,8,789,2.2828,0.0274,MultiClass,0.7659,0.0267,0.9918,14.1684
18,SqrtBalanced,4,304,3.6415,0.1613,MultiClass,0.7628,0.0328,0.9714,1.0691
12,NaN,5,516,1.9096,0.0870,MultiClass,0.7596,0.0112,0.9831,2.2885
2,SqrtBalanced,8,643,6.7990,0.0355,MultiClassOneVsAll,0.7576,0.0279,0.9597,11.7687
17,NaN,6,846,2.9663,0.0276,MultiClassOneVsAll,0.7551,0.0158,0.8452,5.2117
